### The inventory baseline 

The inventory baseline establishes the **Ideal state** of the inventory data.

It tracks the movement of inventory for the simulation timeline which is December 1st, 2025 to June 30th, 2026. 

**Key Logic Constraints**



**Schema example**


In [1]:
import yaml
import random
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

random.seed(cfg["simulation"]["random_seed"])
np.random.seed(cfg["simulation"]["random_seed"])

SIM_START = date(2025, 12, 1)
SIM_END = date(2026,6,30)

## loading the master data
df_supplier = pd.read_csv("dim_supplier.csv")
df_product = pd.read_csv("dim_product.csv")
df_node = pd.read_csv("dim_node.csv")
df_hierarchy = pd.read_csv("dim_hub_hierarchy.csv")
df_scanner = pd.read_csv("dim_scanner.csv")

## lookups for the master data
node_lookup = df_node.set_index("node_id").to_dict("index")
scanner_lookup = df_scanner.groupby("assigned_node_id")["device_id"].apply(list).to_dict()
product_lookup = df_product.set_index("product_id").to_dict("index")
hierarchy_lookup = df_hierarchy.set_index("darkstore_id").to_dict("index")

darkstores = df_node[df_node["node_type"] == "darkstore"].copy()
regional_hubs = df_node[df_node["node_type"] == "regional_hub"].copy()
mother_hubs = df_node[df_node["node_type"] == "mother_hub"].copy()

print(f"Darkstores: {len(darkstores)}")
print(f"Products: {len(df_product)}")
print(f"Sim days: {(SIM_END - SIM_START).days}")

Darkstores: 100
Products: 100
Sim days: 211


event_type options:
- snapshot (stock count at scheduled times, 6:00, 13:00, 18:00, 23:30)
- inbound_transfer (hub sends stock to darkstore)
- lateral_transfer (darkstore sends stick to sibling darkstore)
- customer_fulfillment (units sold, reduces stock)
- rto_return (units returned, increases the stock)

In [2]:
event_schema ={
    "event_id": str, # evt_00001 and so on
    "event_type": str, # one of the 5 event type options
    "event_timestamp": datetime,
    "node_id": str, # the node at which the event is occuring
    "device_id": str, # id of the scanner logging the movement
    "product_id": str,
    "units_sold": int, # value>0 for fulfillment, otherwise 0
    "units_rto": int, # value > 0 for return, otherwise 0
    "units_transferred": int, # value >0 for transfer events, otherwise 0
    "source_node": str, # for transfers where the stock came from
    "destination_node": str, # for transfers, where the stock went
    "stock_on_hand": int, # current stock AFTER this event
    "rto_reason": str, # nullable, only str when rto_return is there
    "processing_lag_hours": float, #normal 0-4, chaos will inject outliers
    "session_id": str, # groups scanner pings from one session
}

In [3]:
# initialise inventory state
# stock_state[node_id][product_id] = current units on hand
stock_state = {}
for _, node in df_node.iterrows():
    stock_state[node["node_id"]] = {}
    for _, prod in df_product.iterrows():
        # start with 60-80% of capcacity allocated across products
        per_product = int(node["capacity_units"]* 0.70/ len(df_product))
        stock_state[node["node_id"]][prod["product_id"]] = max(10, per_product)

events = []
event_num = 1

# loop over each day of the simulation
current_date = SIM_START
while current_date <= SIM_END:
    # for each darkstore
    for _, ds in darkstores.iterrows():
        # skip if not operational yet
        op_since = date.fromisoformat(str(ds['operational_since']))
        if current_date < op_since:
            continue

        # generate snapshot events at scheduled times
        # generate dulfillment events throughout the day
        # check reorder triggers and generate transfers
        # generate RTO events based on rto_rate_base
        pass

    current_date += timedelta(days=1)
df_events = pd.DataFrame(events)

In [4]:
# helper fuctions:
def get_demand_multiplier(current_date, zone_type, node_tier, category, city):
    """
    Returns the demand multiplier for this node/category/date combination.
    Reads demand_spike_events from config.yaml
    Returns 1.0 if no spikes
    """
    best = 1.0
    for spike in cfg["demand_spike_events"]:
        # date match
        if "date" in spike:
            if current_date != date.fromisoformat(str(spike["date"])):
                continue
        elif "date_range" in spike:
            start = date.fromisoformat(spike["date_range"][0])
            end = date.fromisoformat(spike["date_range"][1])
            if not (start <= current_date <= end):
                continue
        else:
            continue
                
        #category match
        cats = spike["product_category"]
        if isinstance(cats, str):
            cats = [cats]
        if category not in cats:
            continue
                
        # scope match
        if "scope" in spike:
            if spike["scope"] == "metropolitan" and node_tier != "metropolitan":
                continue
            if "cities" in spike and city not in spike["cities"]:
                continue

        # zone_type match
        if "zone_type" in spike:
            if zone_type != spike["zone_type"]:
                continue

        best = max(best, spike["multiplier"])

    return best
        

def pick_scanner(node_id, zone_pref=None):
    """
    Returns a device_id for the given node, preferring zone_pref.
    """
    node_scanners = df_scanner[df_scanner["assigned_node_id"] == node_id]
    if zone_pref:
        preferred = node_scanners[node_scanners["scanner_zone"] == zone_pref]
        if len(preferred) > 0:
            return random.choice(preferred["device_id"].tolist())
        if len(node_scanners) > 0:
            return random.choice(node_scanners["device_id"].tolist())
        return f"SCAN_FALLBACK_{node_id}"

def session_id(node_id, current_date, tag):
    """Groups scanner pings from one session together"""
    return f"SES_{node_id}_{current_date.strftime('%Y%m%d')}_{tag}"
        
def base_daily_demand(node_capacity, category, node_tier):
    """
    Approximate units of one product sold per day at this node.
    Perishables turn over faster. Metro nodes move more volume.
    """
    turnover = {
        "dairy": 0.007,
        "beverages": 0.005,
        "staples": 0.003,
        "snacks": 0.004,
        "personal_care": 0.002,
        "household": 0.002,
        "frozen": 0.003,
    }
    tier_scale = {"metropolitan": 1.0, "tier_1": 0.65, "tier_2": 0.35}

    base = (
        node_capacity * turnover.get(category, 0.003) * tier_scale.get(node_tier, 0.5)
    )
    return max(1, int(base))
        
# product lookup for fast access inside loops
prod_cfg = cfg["product_categories"]
product_list = df_product.to_dict("records") # list of dicts, faster than iterrows

RTO_REASONS = [
    "Customer not available",
    "Damaged in transit",
    "Wrong item delivered",
    "Quality issue on arrival",
    "Customer refused delivery",
]

print("Helper functions ready.")

Helper functions ready.


### **SnapShot events**

In [5]:
def generate_snapshots(current_date, node_id, node_type, stock_state, event_counter):
    """
    Generates stock per count snapshots at scheduled times.
    One snapshot row per scheduled time per node (captures total stock). 
    """
    snaps = []
    schedule_key = {
        "darkstore": "darkstore",
        "regional_hub": "regional_hub",
        "mother_hub": "mother_hub",
    }.get(node_type, "darkstore")

    times = cfg["snapshot_schedule"][schedule_key]

    for time_str in times:
        hour, minute = map(int, time_str.split(":"))
        ts = datetime.combine(
            current_date,
            datetime.min.time().replace(hour = hour, minute = minute)
        )

        total_stock = sum(stock_state[node_id].values())
        snaps.append({
            "event_id": f"EVT_{event_counter:07d}",
            "event_type": "snapshot",
            "event_timestamp": ts,
            "node_id": node_id,
            "device_id": pick_scanner(node_id, "stock_count"),
            "product_id": None,
            "units_sold": 0,
            "units_rto": 0,
            "units_transferred": 0,
            "source_node": None,
            "destination_node": None,
            "stock_on_hand": total_stock,
            "rto_reason": None,
            "processing_lag_hours": round(random.uniform(0.0, 2.0), 3),
            "session_id": session_id(node_id, current_date, f"SNAP_{time_str.replace(':','')}"),

        })
        event_counter += 1


    return  snaps, event_counter
test_snaps, _ = generate_snapshots(SIM_START, "DS_AND_01", "darkstore", stock_state,1)
print(f"Snapshots for one node on one day: {len(test_snaps)}")
for s in test_snaps:
    print(f" {s['event_timestamp']} total_stock={s['stock_on_hand']}")

Snapshots for one node on one day: 4
 2025-12-01 06:00:00 total_stock=1000
 2025-12-01 13:00:00 total_stock=1000
 2025-12-01 18:00:00 total_stock=1000
 2025-12-01 23:30:00 total_stock=1000


### **Fulfillments and RTOs**

In [6]:
def generate_fulfillments_and_rtos(current_date, ds_row, stock_state, event_counter):
    """
    For each active product today:
      - generates one fulfillment event (daily aggregated demand)
      - possibly generates one RTO event (based on category rto_rate_base)
    """
    node_id  = ds_row["node_id"]
    capacity = ds_row["capacity_units"]
    tier     = ds_row["tier"]
    zone_type = ds_row["zone_type"]
    city     = ds_row["city"]
    day_events = []

    category_weights = {
        "dairy": 3, "beverages": 3, "snacks": 3,
        "staples": 2, "frozen": 2,
        "personal_care": 1, "household": 1,
    }
    weights = [category_weights.get(p["category"], 1) for p in product_list]
    n_active = random.randint(10, 20)
    active_products = random.choices(product_list, weights=weights, k=n_active)

    for prod in active_products:
        prod_id  = prod["product_id"]
        category = prod["category"]
        current_stock = stock_state[node_id][prod_id]

        if current_stock <= 0:
            continue

        multiplier = get_demand_multiplier(current_date, zone_type, tier, category, city)
        base  = base_daily_demand(capacity, category, tier)
        noise = random.uniform(0.6, 1.4)
        units = min(int(base * multiplier * noise), current_stock)

        if units <= 0:
            continue

        stock_state[node_id][prod_id] -= units

        ts = datetime.combine(
            current_date,
            datetime.min.time().replace(
                hour=random.randint(7, 23),
                minute=random.randint(0, 59)
            )
        )

        day_events.append({
            "event_id":             f"EVT_{event_counter:07d}",
            "event_type":           "customer_fulfillment",
            "event_timestamp":      ts,
            "node_id":              node_id,
            "device_id":            pick_scanner(node_id, "outbound"),
            "product_id":           prod_id,
            "units_sold":           units,
            "units_rto":            0,
            "units_transferred":    0,
            "source_node":          None,
            "destination_node":     None,
            "stock_on_hand":        stock_state[node_id][prod_id],
            "rto_reason":           None,
            "processing_lag_hours": round(random.uniform(0.0, 2.0), 3),
            "session_id":           session_id(node_id, current_date, "FUL"),
        })
        event_counter += 1

        # RTO check
        rto_rate = prod_cfg[category]["rto_rate_base"]
        if random.random() < rto_rate:
            rto_qty = max(1, int(units * random.uniform(0.05, 0.20)))
            stock_state[node_id][prod_id] += rto_qty
            rto_ts = ts + timedelta(hours=random.randint(1, 6))

            day_events.append({
                "event_id":             f"EVT_{event_counter:07d}",
                "event_type":           "rto_return",
                "event_timestamp":      rto_ts,
                "node_id":              node_id,
                "device_id":            pick_scanner(node_id, "returns"),
                "product_id":           prod_id,
                "units_sold":           0,
                "units_rto":            rto_qty,
                "units_transferred":    0,
                "source_node":          None,
                "destination_node":     None,
                "stock_on_hand":        stock_state[node_id][prod_id],
                "rto_reason":           random.choice(RTO_REASONS),
                "processing_lag_hours": round(random.uniform(0.0, 1.0), 3),
                "session_id":           session_id(node_id, current_date, "RTO"),
            })
            event_counter += 1

    return day_events, event_counter, stock_state

print("Fulfillment + RTO function ready.")

Fulfillment + RTO function ready.


next to do: main loop and validation

### **Main Loop**